In [26]:

from pathlib import Path

cuda_code = r"""
#include <cuda_runtime.h>
#include <stdio.h>

__global__ void vector_add(const float* a, const float* b, float* c, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        c[idx] = a[idx] + b[idx];
    }
}

int main() {
    const int N = 1 << 20;
    const size_t bytes = N * sizeof(float);

    float *a, *b, *c;

    cudaMallocManaged(&a, bytes);
    cudaMallocManaged(&b, bytes);
    cudaMallocManaged(&c, bytes);

    for (int i = 0; i < N; i++) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    vector_add<<<blocks, threads>>>(a, b, c, N);
    cudaDeviceSynchronize();

    printf("Result: %f\n", c[0]);

    cudaFree(a);
    cudaFree(b);
    cudaFree(c);

    return 0;
}
"""

Path("vector_add.cu").write_text(cuda_code)
print("Created vector_add.cu")


Created vector_add.cu


In [37]:
!nvcc -O3 -lineinfo vector_add.cu -o vector_add

In [38]:
!ls

sample_data  vector_add  vector_add.cu


In [28]:
!nvidia-smi

Fri Oct 23 03:53:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [29]:
!ncu --mode launch-and-attach

==ERROR== Missing executable to launch.


In [30]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-1ff35a48-fdfc-9e13-a94c-2ab1fba38cc1)


In [31]:
!ncu --version

NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.3.1.0 (build 36398880) (public-release)


In [32]:
!ncu --help

usage: ncu [options] [program] [program-arguments]

General Options:
  -h [ --help ]                         Print this help message.
  -v [ --version ]                      Print the version number.
  --mode arg (=launch-and-attach)       Select the mode of interaction with the target application:
                                          launch-and-attach
                                          (launch and attach for profiling)
                                          launch
                                          (launch and suspend for later attach)
                                          attach
                                          (attach to launched application)
  -p [ --port ] arg (=49152)            Base port for connecting to target application
  --max-connections arg (=64)           Maximum number of ports for connecting to target application
  --config-file arg (=1)                Use config.ncu-cfg config file to set parameters. Searches in the current 
        

In [33]:
!ncu --list-sets

---------- --------------------------------------------------------------------------- ------- -----------------
Identifier Sections                                                                    Enabled Estimated Metrics
---------- --------------------------------------------------------------------------- ------- -----------------
basic      LaunchStats, Occupancy, SpeedOfLight, WorkloadDistribution                  yes     213              
detailed   ComputeWorkloadAnalysis, LaunchStats, MemoryWorkloadAnalysis, MemoryWorkloa no      906              
           dAnalysis_Chart, Occupancy, SourceCounters, SpeedOfLight, SpeedOfLight_Roof                          
           lineChart, WorkloadDistribution                                                                      
full       ComputeWorkloadAnalysis, InstructionStats, LaunchStats, MemoryWorkloadAnaly no      7794             
           sis, MemoryWorkloadAnalysis_Chart, MemoryWorkloadAnalysis_Tables, NumaAffin          

NCU Sets (Nsight Compute Sets) are predefined collections of GPU performance metrics that NVIDIA Nsight Compute measures together.

ELI5 analogy: Imagine taking your car to a mechanic. Instead of checking every single part, you ask for a specific type of inspection—engine performance, fuel efficiency, or a complete inspection. NCU sets work similarly for GPU kernels.

Common NCU Sets

| Set      | What it measures                                                                             |
|----------|---------------------------------------------------------------------------------------------|
| basic    | Essential performance metrics, such as kernel execution time and basic hardware utilization. |
| detailed | More comprehensive information about memory, compute, and GPU execution behavior.            |
| full     | A broad collection of available profiling metrics and analysis sections.                      |

In [34]:
!ncu --list-sets

---------- --------------------------------------------------------------------------- ------- -----------------
Identifier Sections                                                                    Enabled Estimated Metrics
---------- --------------------------------------------------------------------------- ------- -----------------
basic      LaunchStats, Occupancy, SpeedOfLight, WorkloadDistribution                  yes     213              
detailed   ComputeWorkloadAnalysis, LaunchStats, MemoryWorkloadAnalysis, MemoryWorkloa no      906              
           dAnalysis_Chart, Occupancy, SourceCounters, SpeedOfLight, SpeedOfLight_Roof                          
           lineChart, WorkloadDistribution                                                                      
full       ComputeWorkloadAnalysis, InstructionStats, LaunchStats, MemoryWorkloadAnaly no      7794             
           sis, MemoryWorkloadAnalysis_Chart, MemoryWorkloadAnalysis_Tables, NumaAffin          

In [35]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [36]:
!nvcc --help


Usage  : nvcc [options] <inputfile>

Options for specifying the compilation phase
More exactly, this option specifies up to which stage the input files must be compiled,
according to the following compilation trajectories for different input file types:
        .c/.cc/.cpp/.cxx : preprocess, compile, link
        .o               : link
        .i/.ii           : compile, link
        .cu              : preprocess, cuda frontend, PTX assemble,
                           merge with host C code, compile, link
        .gpu             : cicc compile into cubin
        .ptx             : PTX assemble into cubin.

--cuda                                              (-cuda)                     
        Compile all .cu input files to .cu.cpp.ii output.

--cubin                                             (-cubin)                    
        Compile all .cu/.gpu/.ptx input files to device-only .cubin files.  This
        step discards the host code for each .cu input file.

--fatbin          

In [39]:
!ncu --set basic ./vector_add

==PROF== Connected to process 10549 (/content/vector_add)
==PROF== Profiling "vector_add" - 0: 0%....50%....100% - 10 passes
Result: 3.000000
==PROF== Disconnected from process 10549
[10549] vector_add@127.0.0.1
  vector_add(const float *, const float *, float *, int) (4096, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.56
    SM Frequency                    Mhz       584.10
    Elapsed Cycles                cycle       29,482
    Memory Throughput                 %        90.45
    DRAM Throughput                   %        90.45
    Duration                         us        50.46
    L1/TEX Cache Throughput           %        31.73
    L2 Cache Throughput               %        28.64
    SM Active Cycles              cycle  

In [40]:
!ncu --set full -o vector_add_report -f ./vector_add

==PROF== Connected to process 10712 (/content/vector_add)
==PROF== Profiling "vector_add" - 0: 0%....50%....100% - 33 passes
Result: 3.000000
==PROF== Disconnected from process 10712
==PROF== Report: /content/vector_add_report.ncu-rep


In [41]:
!ls

sample_data  vector_add  vector_add.cu	vector_add_report.ncu-rep


Download from remote locally using b64 and Ipython.display

In [44]:

import base64
from pathlib import Path
from IPython.display import display, HTML

file_path = Path("vector_add_report.ncu-rep")
data = base64.b64encode(file_path.read_bytes()).decode()

display(HTML(
    f'<a download="{file_path.name}" '
    f'href="data:application/octet-stream;base64,{data}">'
    f'Download {file_path.name}</a>'
))


Use this to load files to the drive (google drive mount)

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path("vector_add_report.ncu-rep")
destination = Path("/content/drive/MyDrive/GPU-Reports")
destination.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, destination / source.name)

In [45]:
!ncu --set full ./vector_add

==PROF== Connected to process 12108 (/content/vector_add)
==PROF== Profiling "vector_add" - 0: 0%....50%....100% - 33 passes
Result: 3.000000
==PROF== Disconnected from process 12108
[12108] vector_add@127.0.0.1
  vector_add(const float *, const float *, float *, int) (4096, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.56
    SM Frequency                    Mhz       584.41
    Elapsed Cycles                cycle       29,536
    Memory Throughput                 %        90.33
    DRAM Throughput                   %        90.33
    Duration                         us        50.53
    L1/TEX Cache Throughput           %        31.77
    L2 Cache Throughput               %        28.59
    SM Active Cycles              cycle  

In [46]:
!ncu --list-sections

-------------------------------------------- ----------------------------------------------------------------- ------- --------------------------------------------------
Identifier                                   Display Name                                                      Enabled Filename                                          
-------------------------------------------- ----------------------------------------------------------------- ------- --------------------------------------------------
C2CLink                                      C2CLink                                                           no      ...sight Compute/2025.3.1/Sections/C2CLink.section
ComputeWorkloadAnalysis                      Compute Workload Analysis                                         no      ...25.3.1/Sections/ComputeWorkloadAnalysis.section
InstructionStats                             Instruction Statistics                                            no      ...2025.3.1/Sections/Instructio

In [47]:
!ncu --list-rules

------------------------ --------------------------- ------------------------------ --------------------------------------------------
Identifier               Display Name                Description                    Filename                                          
------------------------ --------------------------- ------------------------------ --------------------------------------------------
AchievedOccupancy        Achieved Occupancy          Analysis of the Achieved       ... Compute/2025.3.1/Sections/AchievedOccupancy.py
                                                     Occupancy                                                                        
CPIStall                 Warp Stall                  Warp stall analysis            ...IA Nsight Compute/2025.3.1/Sections/CPIStall.py
Compute                  Compute Bottleneck          Detect bottlenecks arising     ...DIA Nsight Compute/2025.3.1/Sections/Compute.py
                                                     fr

In [56]:
!ncu --list-metrics

smsp__warps_active.sum.peak_sustained
 -smsp__warps_active.sum.peak_sustained
smsp__warps_active.min.peak_sustained
 -smsp__warps_active.min.peak_sustained
smsp__warps_active.max.peak_sustained
 -smsp__warps_active.max.peak_sustained
smsp__warps_active.avg.peak_sustained
 -smsp__warps_active.avg.peak_sustained
smsp__maximum_warps_avg_per_active_cycle
 -smsp__maximum_warps_avg_per_active_cycle
smsp__cycles_elapsed.sum
 -smsp__cycles_elapsed.sum
smsp__cycles_elapsed.min
 -smsp__cycles_elapsed.min
smsp__cycles_elapsed.max
 -smsp__cycles_elapsed.max
smsp__cycles_elapsed.avg
 -smsp__cycles_elapsed.avg
smsp__cycles_active.sum
 -smsp__cycles_active.sum
smsp__cycles_active.min
 -smsp__cycles_active.min
smsp__cycles_active.max
 -smsp__cycles_active.max
smsp__cycles_active.avg
 -smsp__cycles_active.avg
sm__warps_active.sum.per_cycle_active
 -sm__warps_active.sum.per_cycle_active
sm__warps_active.sum.pct_of_peak_sustained_active
 -sm__warps_active.sum.pct_of_peak_sustained_active
sm__warps_active

https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html?utm_source=chatgpt.com#metrics-reference

ncu list metrics, but basic mode

In [63]:
!ncu --set basic --list-metrics 

smsp__warps_active.sum.peak_sustained
 -smsp__warps_active.sum.peak_sustained
smsp__warps_active.min.peak_sustained
 -smsp__warps_active.min.peak_sustained
smsp__warps_active.max.peak_sustained
 -smsp__warps_active.max.peak_sustained
smsp__warps_active.avg.peak_sustained
 -smsp__warps_active.avg.peak_sustained
smsp__maximum_warps_avg_per_active_cycle
 -smsp__maximum_warps_avg_per_active_cycle
smsp__cycles_elapsed.sum
 -smsp__cycles_elapsed.sum
smsp__cycles_elapsed.min
 -smsp__cycles_elapsed.min
smsp__cycles_elapsed.max
 -smsp__cycles_elapsed.max
smsp__cycles_elapsed.avg
 -smsp__cycles_elapsed.avg
smsp__cycles_active.sum
 -smsp__cycles_active.sum
smsp__cycles_active.min
 -smsp__cycles_active.min
smsp__cycles_active.max
 -smsp__cycles_active.max
smsp__cycles_active.avg
 -smsp__cycles_active.avg
sm__warps_active.sum.per_cycle_active
 -sm__warps_active.sum.per_cycle_active
sm__warps_active.sum.pct_of_peak_sustained_active
 -sm__warps_active.sum.pct_of_peak_sustained_active
sm__warps_active

In [58]:
!ncu --query-metrics | grep "dram"

dram__bytes                                                                 Counter         byte            # of bytes accessed in DRAM                                           
dram__bytes_read                                                            Counter         byte            # of bytes read from DRAM                                             
dram__bytes_write                                                           Counter         byte            # of bytes written to DRAM                                            
dram__cycles_active                                                         Counter         cycle           # of cycles where DRAM was active                                     
dram__cycles_active_read                                                    Counter         cycle           # of cycles where DRAM was active for reads                           
dram__cycles_active_write                                                   Counter         cycle        

Run ncu prifiling from device "0"

In [62]:
!ncu --devices 0 ./vector_add

==PROF== Connected to process 16594 (/content/vector_add)
==PROF== Profiling "vector_add" - 0: 0%....50%....100% - 10 passes
Result: 3.000000
==PROF== Disconnected from process 16594
[16594] vector_add@127.0.0.1
  vector_add(const float *, const float *, float *, int) (4096, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.59
    SM Frequency                    Mhz       584.36
    Elapsed Cycles                cycle       29,422
    Memory Throughput                 %        90.08
    DRAM Throughput                   %        90.08
    Duration                         us        50.34
    L1/TEX Cache Throughput           %        31.70
    L2 Cache Throughput               %        28.70
    SM Active Cycles              cycle  

In [64]:
!ncu --list-chips

ad102, ad103, ad104, ad106, ad107, ga100, ga102, ga103, ga104, ga106, ga107, ga10b, gb100, gb102, gb10b, gb110, gb202, gb203, gb205, gb206, gb207, gb20b, gh100, tu102, tu104, tu106, tu116, tu117


In [80]:
!ncu --section LaunchStats -o launch_stats -f ./vector_add

==PROF== Connected to process 19967 (/content/vector_add)
==PROF== Profiling "vector_add" - 0: 0%....50%....100% - 1 pass
Result: 3.000000
==PROF== Disconnected from process 19967
==PROF== Report: /content/launch_stats.ncu-rep


In [81]:
!ncu --import launch_stats.ncu-rep --csv --page details --log-file launch_stats.csv 

| Argument      | What it does                                         | Simple explanation                                                                              |
|---------------|------------------------------------------------------|-------------------------------------------------------------------------------------------------|
| --import      | Loads an existing .ncu-rep profiling report.         | Read previously collected GPU profiling data instead of running the kernel again.               |
| --csv         | Formats profiling results as CSV (Comma-Separated Values). | Make the results readable by Excel, pandas, or other data tools.                         |
| --page details| Selects the detailed, section-based profiling report.| Show the collected performance metrics organized into sections such as Launch Statistics.       |
| --log-file    | Writes NCU's report output to the specified file.    | Save the results to launch_stats.csv instead of displaying them in the terminal.               |

In [86]:
!ncu --help | grep -A 15 -i "page"

  --page arg (=details)                 Select report page to output:
                                          details: sections and rules
                                          raw: all collected metrics
                                          source: source code
                                          session: session and device attributes
  --print-source arg                    Select source view:
                                          sass
                                          ptx
                                          cuda
                                          cuda,sass
                                         Metric correlation with source is available in sass, and cuda,sass source 
                                        view.
                                         Metrics specified with --metrics and specified section file with --section are
                                        correlated.
                                         Consider restricting 

In [89]:
!ncu --help | grep -A 15 -i "log-file"

  --log-file arg                        Send all tool output to the specified file, or
                                          one of the standard channels. The file will be overwritten.
                                          If the file doesn't exist, a new one will be created.
                                          "stdout" as the whole file name indicates standard output
                                             channel (stdout). (default)
                                          "stderr" as the whole file name indicates standard error
                                             channel (stderr).
  -o [ --export ] arg                   Set the output file for writing the profile results. If not set, a temporary 
                                        file will be used which is removed afterwards.
  -f [ --force-overwrite ]              Force overwriting all output, section or config files (any existing files will 
                                        be overwritten)

In [87]:
!ncu --import launch_stats.ncu-rep --csv --page raw --log-file launch_stats_raw.csv 

In [82]:
import pandas as pd

df = pd.read_csv('launch_stats.csv')
df.head()


,ID,Process ID,Process Name,Host Name,Kernel Name,Context,Stream,Block Size,Grid Size,Device,CC,Section Name,Metric Name,Metric Unit,Metric Value,Rule Name,Rule Type,Rule Description,Estimated Speedup Type,Estimated Speedup
0,0,19967,vector_add,127.0.0.1,"vector_add(const float *, const float *, float...",1,7,"(256, 1, 1)","(4096, 1, 1)",0,7.5,Launch Statistics,Block Size,NaN,256,NaN,NaN,NaN,NaN,NaN
1,0,19967,vector_add,127.0.0.1,"vector_add(const float *, const float *, float...",1,7,"(256, 1, 1)","(4096, 1, 1)",0,7.5,Launch Statistics,Function Cache Configuration,NaN,CachePreferNone,NaN,NaN,NaN,NaN,NaN
2,0,19967,vector_add,127.0.0.1,"vector_add(const float *, const float *, float...",1,7,"(256, 1, 1)","(4096, 1, 1)",0,7.5,Launch Statistics,Grid Size,NaN,"4,096",NaN,NaN,NaN,NaN,NaN
3,0,19967,vector_add,127.0.0.1,"vector_add(const float *, const float *, float...",1,7,"(256, 1, 1)","(4096, 1, 1)",0,7.5,Launch Statistics,Registers Per Thread,register/thread,16,NaN,NaN,NaN,NaN,NaN
4,0,19967,vector_add,127.0.0.1,"vector_add(const float *, const float *, float...",1,7,"(256, 1, 1)","(4096, 1, 1)",0,7.5,Launch Statistics,Shared Memory Configuration Size,Kbyte,32.77,NaN,NaN,NaN,NaN,NaN


In [88]:
import pandas as pd

df = pd.read_csv('launch_stats_raw.csv')
df.head()


,ID,Process ID,Process Name,Host Name,Kernel Name,Context,Stream,Block Size,Grid Size,Device,...,profiler__perfworks_session_reuse,profiler__replayer_passes,profiler__replayer_passes_type_warmup,sm__maximum_warps_avg_per_active_cycle,sm__maximum_warps_per_active_cycle_pct,sm__warps_active.avg.pct_of_peak_sustained_active,sm__warps_active.max.pct_of_peak_sustained_active,sm__warps_active.min.pct_of_peak_sustained_active,sm__warps_active.sum.pct_of_peak_sustained_active,smsp__maximum_warps_avg_per_active_cycle
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,pass,pass,warp,%,%,%,%,%,warp
1,0.0,19967.0,vector_add,127.0.0.1,"vector_add(const float *, const float *, float...",1.0,7.0,"(256, 1, 1)","(4096, 1, 1)",0.0,...,0.0,1.000000,0,32.000000,100.000000,97.358386,99.410298,95.190282,97.358386,8.000000


You can't do this in remote (need to do this in SSH with X11 fowarding back to local)

In [90]:
!ncu --import launch_stats.ncu-rep --open-in-ui 

qt.qpa.plugin: From 6.5.0, xcb-cursor0 or libxcb-cursor0 is needed to load the Qt xcb platform plugin.
qt.qpa.plugin: Could not load the Qt platform plugin "xcb" in "" even though it was found.
This application failed to start because no Qt platform plugin could be initialized. Reinstalling the application may fix this problem.

Available platform plugins are: wayland, xcb, wayland-egl, offscreen.

Application could not be initialized!
    This is likely due to missing Qt platform dependencies.
    For a list of dependencies, please refer to https://doc.qt.io/qt-6/linux-requirements.html
    To view missing libraries, set QT_DEBUG_PLUGINS=1 and re-run the application.

qt.qpa.plugin: From 6.5.0, xcb-cursor0 or libxcb-cursor0 is needed to load the Qt xcb platform plugin.
qt.qpa.plugin: Could not load the Qt platform plugin "xcb" in "" even though it was found.
This application failed to start because no Qt platform plugin could be initialized. Reinstalling the application may fix this p

In [94]:
!ncu --import launch_stats.ncu-rep --print-source sass --page source

------------------ ------------------------------------------------------
Kernel Name        vector_add(const float *, const float *, float *, int)
------------------ ------------------------------------------------------
Address            Source                                                
0x780dc125a300          MOV R1, c[0x0][0x28]                             
0x780dc125a310          S2R R6, SR_CTAID.X                               
0x780dc125a320          S2R R3, SR_TID.X                                 
0x780dc125a330          IMAD R6, R6, c[0x0][0x0], R3                     
0x780dc125a340          ISETP.GE.AND P0, PT, R6, c[0x0][0x178], PT       
0x780dc125a350     @P0  EXIT                                             
0x780dc125a360          MOV R7, 0x4                                      
0x780dc125a370          IMAD.WIDE R4, R6, R7, c[0x0][0x168]              
0x780dc125a380          IMAD.WIDE R2, R6, R7, c[0x0][0x160]              
0x780dc125a390          LDG.E.SYS R4, 

In [95]:
!ncu --import launch_stats.ncu-rep --print-details header

[19967] vector_add@127.0.0.1
  vector_add(const float *, const float *, float *, int) (4096, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                   256
    Function Cache Configuration                     CachePreferNone
    Grid Size                                                  4,096
    Registers Per Thread             register/thread              16
    Shared Memory Configuration Size           Kbyte           32.77
    Driver Shared Memory Per Block        byte/block               0
    Dynamic Shared Memory Per Block       byte/block               0
    Static Shared Memory Per Block        byte/block               0
    # SMs                                         SM        

In [97]:
!ncu --import launch_stats.ncu-rep --print-metric-name label-name

[19967] vector_add@127.0.0.1
  vector_add(const float *, const float *, float *, int) (4096, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Launch Statistics
    ---------------------------------------------------------------------- --------------- ---------------
    Metric Name                                                                Metric Unit    Metric Value
    ---------------------------------------------------------------------- --------------- ---------------
    Block Size (launch__block_size)                                                                    256
    Function Cache Configuration (launch__func_cache_config)                               CachePreferNone
    Grid Size (launch__grid_size)                                                                    4,096
    Registers Per Thread (launch__registers_per_thread)                    register/thread              16
    Shared Memory Configuration Size (launch__shared_mem_config_size) 